## 1. Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p /content/drive/MyDrive/ecri_neural_network_runs
!ln -sfn /content/drive/MyDrive/ecri_neural_network_runs results

## 2. Data

In [3]:
!git clone --depth 1 https://github.com/ecri-data/ECRI-Dataset.git /content/ecri
!mkdir -p data notebooks
!cp $(find /content/ecri -name "ECRIDataSet.csv") $(find /content/ecri -name "ECRIRawData.csv") data/
!ls data

Cloning into '/content/ecri'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 17 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 1.32 MiB | 5.78 MiB/s, done.
ECRIDataSet.csv  ECRIRawData.csv


## 3. Scripts (upload the four `.py` files into `notebooks/`)

In [4]:
!ls notebooks
!python notebooks/Supriya_Neural_network_implementation.py --help | grep -q "select-on-test" && grep -q 'default="assessments"' notebooks/Supriya_Neural_network_implementation.py && echo "scripts: OK" || echo "WRONG VERSION: upload the latest scripts from the repository"

Supriya_kNN_extension.py		  Supriya_Neural_Network_REPORT.md
Supriya_Neural_network_implementation.py  Supriya_robustness_extension.py
scripts: OK


## 4. Quick check

In [5]:
!python notebooks/Supriya_robustness_extension.py --raw-dataset data/ECRIRawData.csv --output-dir /content/robust_smoke --seeds 42 --groups school --targets RMwidRS_target --epochs 3 --pretrain-epochs 2 --mask-rates 0 0.25 --tests-missing 1 --label-fractions 0.1 1.0 | tail -5
!ls /content/robust_smoke && rm -rf /content/robust_smoke

  missing_tests       1.00 intervention MaskMLP_triplet balanced_accuracy  0.513                            3  MLP_indicator                 0.010                          2                        0 LogisticRegression                  -0.063                  0.016      5
  missing_tests       1.00 intervention  MaskMLP_maskft balanced_accuracy  0.510                            3  MLP_indicator                 0.007                          1                        1 LogisticRegression                  -0.066                  0.013      5
  missing_tests       1.00 intervention  MaskMLP_frozen balanced_accuracy  0.500                            4  MLP_indicator                -0.003                          1                        1 LogisticRegression                  -0.076                  0.002      5
  missing_tests       1.00 intervention   MaskMLP_lowlr balanced_accuracy  0.500                            4  MLP_indicator                -0.003                          1           

In [ ]:
!python notebooks/Supriya_Neural_network_implementation.py --dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv --output-dir /content/smoke --models neural --group school --target RMwidRS_target --epochs 3 --pretrain-epochs 2 --no-tsne | tail -3
!rm -rf /content/smoke

school RMwidRS_target       MaskMLP          all       0.525903                0.498762     0.563791
school RMwidRS_target       MaskMLP intervention       0.569485                0.501312     0.641909
Figures written to /content/smoke/figures


## 5. Experiments

In [ ]:
%%bash
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1
BASE="--dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv"
run() {
  name=$1; done_file=$2; shift 2
  if [ -f "results/neural_network/$name/$done_file" ]; then echo "skip $name"; return; fi
  python notebooks/Supriya_Neural_network_implementation.py $BASE --output-dir results/neural_network/$name "$@" \
    > log_${name//\//_}.txt 2>&1 && echo "done $name" || echo "FAILED $name (see log_${name//\//_}.txt)"
}
throttle() { while (( $(jobs -rp | wc -l) >= 2 )); do wait -n; done; }

throttle; run main loss_ablation.csv --seed 42 --importance --loss-ablation &
for s in 43 44 45 46;    do throttle; run seeds/s$s results.csv --seed $s --no-tsne & done
for s in 42 43 44 45 46; do throttle; run train/s$s results.csv --pretrain-scope train --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run all946/s$s results.csv --eval-rows all --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run complete/all-s$s results.csv --eval-rows complete --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run complete/train-s$s results.csv --eval-rows complete --pretrain-scope train --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run simsiam/s$s results.csv --pretrain-loss simsiam --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run contrastive/s$s results.csv --pretrain-loss contrastive --seed $s --no-tsne & done
for e in 5 10;           do throttle; run pretrain_epochs/e$e results.csv --pretrain-epochs $e --seed 42 --no-tsne & done
throttle; run diagnostic_test_selection results.csv --select-on-test --seed 42 --no-tsne &
wait
echo "ALL FINISHED"

done seeds/s43
done seeds/s44
done seeds/s45
done seeds/s46
done train/s42
done train/s43
done train/s44
done train/s45
done train/s46
done all946/s42
done all946/s43
done all946/s44
done complete/all-s42
done main
done complete/all-s43
done complete/all-s44
done complete/train-s42
done complete/train-s43
done simsiam/s42
done complete/train-s44
done simsiam/s43
done simsiam/s44
done contrastive/s42
done contrastive/s43
done contrastive/s44
done pretrain_epochs/e5
done pretrain_epochs/e10
done diagnostic_test_selection
ALL FINISHED


## 6. kNN extension

In [ ]:
!python notebooks/Supriya_kNN_extension.py --raw-dataset data/ECRIRawData.csv | tail -12

                       accuracy  balanced_accuracy  pr_auc
model                                                     
LogisticRegression        0.609              0.593   0.663
XGBoost                   0.604              0.593   0.674
MLP_mean                  0.596              0.576   0.655
MLP_zero                  0.591              0.568   0.647
MLP_indicator             0.586              0.562   0.647
MaskMLP                   0.579              0.547   0.638
kNN_raw                   0.572              0.546   0.616
kNN_MaskMLP_embedding     0.515              0.504   0.575
Comparison written to results/knn_extension
Saved kNN results to results/knn_extension


## 7. Robustness extension (missing inputs, scarce labels)

In [6]:
!OMP_NUM_THREADS=1 python notebooks/Supriya_robustness_extension.py --raw-dataset data/ECRIRawData.csv | tail -30

missing_at_test       0.50          all MaskMLP_triplet balanced_accuracy  0.511                            6       MLP_mean                -0.028                         16                       43            XGBoost                  -0.016                 -0.025     60
missing_at_test       0.50          all  MaskMLP_maskft balanced_accuracy  0.513                            5       MLP_mean                -0.026                         19                       38            XGBoost                  -0.014                 -0.023     60
missing_at_test       0.50          all  MaskMLP_frozen balanced_accuracy  0.499                            6       MLP_mean                -0.039                         10                       45            XGBoost                  -0.027                 -0.037     60
missing_at_test       0.50          all   MaskMLP_lowlr balanced_accuracy  0.502                            6       MLP_mean                -0.036                         11           

## 8. Report tables

In [ ]:
!python notebooks/Supriya_Neural_network_report.py --outputs results/neural_network

Runs found: all946/s42, all946/s43, all946/s44, complete/all-s42, complete/all-s43, complete/all-s44, complete/train-s42, complete/train-s43, complete/train-s44, contrastive/s42, contrastive/s43, contrastive/s44, diagnostic_test_selection, main, pretrain_epochs/e10, pretrain_epochs/e5, seeds/s43, seeds/s44, seeds/s45, seeds/s46, simsiam/s42, simsiam/s43, simsiam/s44, train/s42, train/s43, train/s44, train/s45, train/s46
Report written to results/neural_network/report (primary setting: loss=cosine, pretrain=all, rows=assessments, variant=standard)


## 9. Download

In [ ]:
!zip -qr /content/neural_network_results.zip results/neural_network results/knn_extension results/robustness_extension
from google.colab import files
files.download("/content/neural_network_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>